<a href="https://colab.research.google.com/github/vivek-chandan/Youtube-Chatbot-using-RAG-Langchain/blob/main/YOUTUBE_Chatbot_rag_using_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "Paste API HERE"

## Install libraries

In [ ]:
!pip install -U langchain-text-splitters langchain-openai langchain-community faiss-cpu

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

## Step 1a - Indexing (Document Ingestion)

In [ ]:
!pip install youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 13.0 MB/s eta 0:00:00


In [ ]:

from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
print(dir(YouTubeTranscriptApi)) # Verify 'list_transcripts' is now there

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'fetch', 'list']


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "Gfr50f6ZBvo"

try:
    # Initialize the API client
    api = YouTubeTranscriptApi()

    # Fetch the transcript data
    transcript_data = api.fetch(video_id)

    # 1. Format into the list of dictionaries
    formatted_transcript = [
        {
            'text': t.text,
            'start': t.start,
            'duration': t.duration
        }
        for t in transcript_data
    ]

    # 2. Create the flattened string
    transcript_text = " ".join([t.text for t in transcript_data])

    # Print the result to verify
    import json
    print(json.dumps(formatted_transcript[:2], indent=2))
    print("\nTranscript preview:", transcript_text[:100] + "...")

except Exception as e:
    print(f"Error: {e}")

[
  {
    "text": "the following is a conversation with",
    "start": 0.08,
    "duration": 3.44
  },
  {
    "text": "demus hasabis",
    "start": 1.76,
    "duration": 4.96
  }
]

Transcript preview: the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has...


In [ ]:
transcript = formatted_transcript

## Step 1b - Indexing (Text Splitting)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# Using transcript_text which is the string version of the transcript
chunks = splitter.create_documents([transcript_text])

In [ ]:
len(chunks)

168

In [ ]:
chunks[10]

Document(metadata={}, page_content="to the ai agents of the future anyway so going back to your to your journey when did you fall in love with programming first well it's pretty uh pretty young age actually so um you know i started off uh actually games was my first love so starting to play chess when i was around four years old and then um it was actually with winnings from a chess competition that i managed to buy my first chess computer when i was about eight years old it was a zx spectrum which was hugely popular in the uk at the time and uh it's amazing machine because i think it trained a whole generation of programmers in the uk because it was so accessible you know you literally switched it on and there was the basic prompt and you could just get going and um my parents didn't really know anything about computers so but because it was my money from a chess competition i could i could say i i wanted to buy it uh and then you know i just went to bookstores got books on programmin

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: Paste AP**HERE. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['2436bdb8-3f5f-49c6-8915-0c654c888700'])

## Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

In [ ]:
retriever.invoke('What is deepmind')

## Step 3 - Augmentation

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [ ]:
retrieved_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

## Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')